> ### Note on Labs and Assignments:
>
> 🔧 Look for the **wrench emoji** 🔧 — it highlights where you're expected to take action!
>
> These sections are graded and are not optional.
>

# Lab 11: Linear Regression for HR

In this lab, we’ll use the merged HR dataset to build a **linear regression model** to predict the number of years that an employee will work for the company (job tenure).

### Analytics Objectives:
1. Load and explore the dataset
2. Clean and prepare features
3. Encode categorical variables
4. Split the data into training and test sets
5. Standardize features without leaking information from the test set
6. Train and evaluate a baseline linear regression model
7. Engineer features to improve model fit
8. Use regularization for automated feature selection

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Labs/lab_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## HR Turnover - Business Context
A large company employs, at any given point of time, around 4000 employees. However, every year, around 15% of its employees leave the company and need to be replaced with the talent pool available in the job market. The management believes that this employee turnover is bad for the company, because of the following reasons -

- The former employees’ projects get delayed, which makes it difficult to meet timelines, resulting in a reputation loss among consumers and partners
- A sizeable department has to be maintained, for the purposes of recruiting new talent
- More often than not, the new employees have to be trained for the job and/or given time to acclimatise themselves to the company

Hence, the management has contracted an HR analytics firm to understand what factors they should focus on, in order to increase the number of years that employees stay with the company. In other words, they want to know what changes they should make to their workplace, in order to get most of their employees to stay. Also, they want to know which of these variables is most important and needs to be addressed right away.

Since you are one of the star analysts at the firm, this project has been given to you.

You are required to model the number of years employees work for the company using a linear regression. The results thus obtained will be used by the management to understand what changes they should make to their workplace, in order to get most of their employees to stay.

**Target Variable:** `YearsAtCompany` 

## Data Dictionary

Source: [Kaggle HR Analytics Case Study](https://www.kaggle.com/datasets/vjchoudhary7/hr-analytics-case-study)

| Variable                      | Type        | Description |
|-------------------------------|-------------|-------------|
| `Age`                         | Numeric     | Age of the employee |
| `Attrition`                   | Categorical | Whether the employee has left the company (Yes/No) |
| `BusinessTravel`              | Categorical | Frequency of business travel |
| `Department`                  | Categorical | Department name |
| `DistanceFromHome`           | Numeric     | Distance from home to work (in km) |
| `Education`                  | Ordinal     | Employee education level (1–5) |
| `EducationField`             | Categorical | Field of education |
| `EmployeeID`                 | Identifier  | Unique identifier for employee |
| `EmployeeCount`              | Constant    | Always 1 (not useful for modeling) |
| `EnvironmentSatisfaction`    | Ordinal     | Satisfaction with the environment (1–4) |
| `Gender`                     | Categorical | Gender of the employee |
| `JobInvolvement`             | Ordinal     | Level of involvement with job (1–4) |
| `JobLevel`                   | Ordinal     | Employee level (1–5) |
| `JobRole`                    | Categorical | Job title |
| `JobSatisfaction`            | Ordinal     | Satisfaction with the job (1–4) |
| `MaritalStatus`              | Categorical | Marital status |
| `MonthlyIncome`              | Numeric     | Monthly salary in USD |
| `NumCompaniesWorked`         | Numeric     | Number of companies previously worked for |
| `Over18`                     | Constant    | Always "Y" (not useful) |
| `PercentSalaryHike`          | Numeric     | Percentage salary increase |
| `PerformanceRating`          | Ordinal     | Performance rating (1–4) |
| `StandardHours`              | Constant    | Always 80 (not useful) |
| `StockOptionLevel`           | Ordinal     | Stock options level (0–3) |
| `TotalWorkingYears`          | Numeric     | Total years of professional experience |
| `TrainingTimesLastYear`      | Numeric     | Number of training sessions attended last year |
| `WorkLifeBalance`            | Ordinal     | Work-life balance rating (1–4) |
| `YearsAtCompany`             | Numeric     | Years spent at the current company |
| `YearsInCurrentRole`         | Numeric     | Years spent in current role |
| `YearsSinceLastPromotion`    | Numeric     | Years since last promotion |
| `YearsWithCurrManager`       | Numeric     | Years with current manager |
| `JobSatisfaction`            | Ordinal     | Self-reported job satisfaction (1–4) |
| `EnvironmentSatisfaction`    | Ordinal     | Satisfaction with the work environment (1–4) |
| `WorkLifeBalance`            | Ordinal     | Work-life balance rating (1–4) |
| `JobInvolvement`             | Ordinal     | Employee’s job involvement level (1–4) |
| `PerformanceRating`          | Ordinal     | Most recent performance rating |



## Part 1: Load data and packages




In [ ]:
import pandas as pd

# Load the merged HR dataset
url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/merged_hr_data.csv"
df = pd.read_csv(url)

# Preview structure
print("Shape:", df.shape)
df.head()


## Part 2: Data Cleaning

Real-world HR data often contains administrative fields (e.g., ID numbers), constants (same value for all rows), or missing values.

### What We’re Doing:
- Remove irrelevant or constant columns: `EmployeeCount`, `Over18`, `StandardHours`, `EmployeeID`
- Drop rows with missing data

### Why It Matters:
- Non-informative or redundant features can reduce model accuracy and interpretability.
- Regression does not handle missing values natively, so we need a clean dataset.
- Dropping some rows is reasonable here due to the relatively small number of nulls.

> Ethical Note: In practice, dropping rows may disproportionately exclude certain groups—so this step should be handled with caution.


In [ ]:
# Drop unnecessary columns
drop_cols = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeID', 'Attrition']
df.drop(columns=drop_cols, inplace=True)

# Drop rows with any missing values
df.dropna(inplace=True)

# Check result
print("After cleaning:", df.shape)


## Part 3: Encode Categorical Variables

Machine learning algorithms like linear regression require **numeric inputs**. To use categorical data like `Gender` or `JobRole`, we convert them into **dummy variables** using one-hot encoding.

### Key Steps:
- Use `pd.get_dummies()` with `drop_first=True` to avoid multicollinearity

### Why It Matters:
- Ensures model can interpret categorical inputs numerically
- Dropping the first dummy prevents the "dummy variable trap" where one variable is a linear combination of others
- Accurate encoding helps ensure model fairness and interpretability

> Reminder: Avoid encoding identifiers or columns with too many unique levels without reduction.



In [ ]:
# One-hot encode categorical features
df_encoded = pd.get_dummies(df, drop_first=True)

# Preview encoded columns
df_encoded.info()

### 🔧 Do the following - Part 3

1. How many new columns were created during one-hot encoding?  
2. Why is it important to avoid including columns like `EmployeeID` in modeling?
3. Our model is trying to predict employment longevity.  Why is `Attrition` problematic for predicting years at the company? 

Write a few sentences on each of the questions above. No coding is required here. 


🔧 Add comment here:

## Part 4: Train-Test Split

We will split the data into an 80% training set and a 20% test set. The training set is used to fit models and preprocessing steps; the test set remains untouched until evaluation.

> **Important:** Split before applying any preprocessing that learns summary statistics from the data, such as standardization, normalization, imputation, or feature selection. Otherwise, information from the test set can leak into training.

Because `YearsAtCompany` is continuous, we will use a random split rather than stratifying on the target.


In [ ]:
from sklearn.model_selection import train_test_split

# Separate raw features from the target, then split before scaling
X_raw = df_encoded.drop(columns=['YearsAtCompany'])
y = df_encoded['YearsAtCompany']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=42
)

print('Training rows:', X_train_raw.shape[0])
print('Test rows:', X_test_raw.shape[0])

### 🔧 Do the following - Part 4

Why should the test set be kept separate before we fit a scaler, imputer, or feature-selection procedure?

**Think about this:**
**Write one or two sentences** explaining what information leakage is and why it makes test-set performance less trustworthy.


🔧 Add comment here:

## Part 5: Standardize Features Without Leakage

Linear regression can be fit without standardization, but scaling puts features on comparable units and is required for the regularized model later in the lab.

We **fit** `StandardScaler` on the training data only, then use that same fitted scaler to transform the training and test sets. The target variable, `YearsAtCompany`, is not scaled.


In [ ]:
from sklearn.preprocessing import StandardScaler

# Learn the scaling values from training data only
scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw), columns=X_train_raw.columns, index=X_train_raw.index
)
X_test = pd.DataFrame(
    scaler.transform(X_test_raw), columns=X_test_raw.columns, index=X_test_raw.index
)

X_train.head()

### 🔧 Do the following - Part 5

1. In the code cell below, calculate the average `YearsAtCompany` for the training set and the test set.
2. Why might their averages be slightly different?

In [ ]:
# 🔧 Add code here

### Reflection: 
1. Why do we fit the scaler using `X_train_raw` instead of using all rows?

#### ✍️ Your Response: 🔧
1. 

## Part 6: Train the Regression

Now we fit a linear regression model using the training data.  

In [ ]:
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the Linear Regression model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Display standardized coefficients from the baseline model
coefficients_linear = pd.Series(linear_model.coef_, index=X_train.columns)
print("\nLinear Regression Coefficients (ordered):")
print(coefficients_linear.sort_values(ascending=False))


### 🔧 Do the following - Part 6

1. Which features are most positively associated with high job tenure (years at company)?
2. Which features are most negatively associated with staying?

### Reflection: 
Write a few sentences on each of the questions above. No coding is required here. 

### ✍️ Your Response: 🔧
1. 
2. 

## Part 7: Evaluate Baseline Model Performance

Let’s compare the baseline model's training and test performance. We'll compute:
- Mean Squared Error (MSE)
- R-Squared

In [ ]:
# Evaluate both training and test performance
y_pred_train_linear = linear_model.predict(X_train)
y_pred_linear = linear_model.predict(X_test)

train_r2 = r2_score(y_train, y_pred_train_linear)
mse = mean_squared_error(y_test, y_pred_linear)
r2 = r2_score(y_test, y_pred_linear)

# Print the evaluation metrics
print(f"Training R-squared (R2): {train_r2:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

Now visualize the model output

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a scatter plot of predicted vs actual YearsAtCompany
plt.figure(figsize=(10, 6))
ax = sns.regplot(x=y_test, y=y_pred_linear, scatter_kws={'alpha':0.6}, line_kws={"color": "red"})
ax.set_xlabel("Actual YearsAtCompany")
ax.set_ylabel("Predicted YearsAtCompany")
plt.title("Actual vs. Predicted YearsAtCompany with Regression Line")
plt.grid(True)
plt.show()

### 🔧 Reflection - Part 7

1. The baseline model should produce training and test R-squared values of roughly 0.76 and 0.75. What does the small gap between them suggest about overfitting versus underfitting?
2. Why is this still an underfitting model, even though the two scores are similar?
3. What kind of change could help capture relationships that a simple linear model misses?

### ✍️ Your Response: 🔧
1. 
2. 

## Part 8: Feature Engineering to Improve Model Fit

The similar training and test R-squared values indicate that the baseline model is not memorizing the training data. Instead, it is likely **underfitting**: its linear terms are not flexible enough to capture all meaningful patterns.

Before removing predictors, we should first give the model richer information. **Feature engineering** creates useful transformations and interactions from existing predictors. Below, we add a few defensible nonlinear and interaction features. Each feature is created independently within a row, so it does not learn summary statistics from the test set.

> A feature should be available at the time the prediction would be made. Avoid creating features that use `YearsAtCompany` itself or any information known only after the employee's tenure has ended.


In [ ]:
import numpy as np

def engineer_features(data):
    """Add nonlinear and interaction terms using predictors available at prediction time."""
    engineered = data.copy()
    engineered['LogMonthlyIncome'] = np.log1p(engineered['MonthlyIncome'])
    engineered['LogDistanceFromHome'] = np.log1p(engineered['DistanceFromHome'])
    engineered['TotalWorkingYearsSquared'] = engineered['TotalWorkingYears'] ** 2
    engineered['ExperienceByJobLevel'] = engineered['TotalWorkingYears'] * engineered['JobLevel']
    engineered['SatisfactionInteraction'] = (
        engineered['JobSatisfaction'] * engineered['EnvironmentSatisfaction']
    )
    return engineered

# Engineer features after the split, then fit a new scaler on the training data only
X_train_engineered_raw = engineer_features(X_train_raw)
X_test_engineered_raw = engineer_features(X_test_raw)

engineered_scaler = StandardScaler()
X_train_engineered = pd.DataFrame(
    engineered_scaler.fit_transform(X_train_engineered_raw),
    columns=X_train_engineered_raw.columns, index=X_train_engineered_raw.index
)
X_test_engineered = pd.DataFrame(
    engineered_scaler.transform(X_test_engineered_raw),
    columns=X_test_engineered_raw.columns, index=X_test_engineered_raw.index
)

engineered_model = LinearRegression()
engineered_model.fit(X_train_engineered, y_train)
engineered_train_r2 = r2_score(y_train, engineered_model.predict(X_train_engineered))
engineered_test_r2 = r2_score(y_test, engineered_model.predict(X_test_engineered))

print(f'Engineered-model training R2: {engineered_train_r2:.2f}')
print(f'Engineered-model test R2: {engineered_test_r2:.2f}')

### 🔧 Do the following – Part 8

1. Add two additional features to the `engineer_features` function. Explain why each might help predict tenure.
2. Rerun the engineered model. Compare its training and test R-squared values with the baseline model.
3. Did your additions improve test performance, increase overfitting, or neither?


In [ ]:
# 🔧 Add your two engineered features inside engineer_features(), then rerun cell 31.
# Keep each new feature based only on predictor columns—not on YearsAtCompany.

# Example pattern:
# engineered['NewFeatureName'] = engineered['ExistingColumn1'] * engineered['ExistingColumn2']

## Part 9: Regularization for Feature Selection

Selecting variables manually from the coefficients of the same model we evaluated is not a reliable feature-selection process. Coefficients can change with correlated predictors, and choosing features after looking at them can overstate performance.

Instead, we will use **Lasso regression with cross-validation**. Lasso adds a penalty that can shrink unhelpful coefficients all the way to zero. `LassoCV` chooses the penalty strength using folds from the training data only, then we evaluate the selected model once on the untouched test set.

> Scaling is essential for Lasso because the penalty depends on coefficient size. We already scaled the engineered features using the training set only.


In [ ]:
from sklearn.linear_model import LassoCV

# Choose the L1 penalty using training-data cross-validation only
lasso = LassoCV(cv=5, random_state=42, max_iter=20000)
lasso.fit(X_train_engineered, y_train)

# Features with exactly zero coefficients were excluded by Lasso
lasso_coefficients = pd.Series(lasso.coef_, index=X_train_engineered.columns)
selected_features = lasso_coefficients[lasso_coefficients != 0].sort_values(key=np.abs, ascending=False)

lasso_train_r2 = r2_score(y_train, lasso.predict(X_train_engineered))
lasso_test_r2 = r2_score(y_test, lasso.predict(X_test_engineered))

print(f'Features retained by Lasso: {len(selected_features)} of {X_train_engineered.shape[1]}')
print(f'Lasso training R2: {lasso_train_r2:.2f}')
print(f'Lasso test R2: {lasso_test_r2:.2f}')
print('\nSelected features and coefficients:')
print(selected_features)

# 🔧 Reflection:
# 1. How does Lasso's test R2 compare with the engineered linear model?
# 2. Why is Lasso-based selection preferable to manually selecting features
#    from the coefficients of the baseline model?

## Export Your Notebook to Submit in Canvas
- Use the instructions from Lab 1

In [ ]:
!jupyter nbconvert --to html "lab_11_regression.ipynb"